# Chapitre 5 · Un peu de hasard

Notebook du chapitre 5 de *Construire un LLM de zéro*. Tu démontes les trois
dernières lignes mystérieuses du programme du chapitre 1 : le **softmax** (et sa
molette `temperature`), la **cross-entropy** (la loss qui valait 4.42), et
l'**échantillonnage** (le tirage au sort du caractère suivant).

**Comment travailler.** La leçon d'abord : tout le code du chapitre, complet et
prêt à exécuter, notion par notion. Lis, exécute, modifie pour voir. En bonus de
fin de leçon, on rebranche les fonctions maison sur le vrai MiniLM du chapitre 1.
À la fin, la section **Exercices** : quatre défis à trous, du plus simple au plus
costaud, validés par des `assert`.

Tout tourne **sans GPU et sans connexion internet** : PyTorch et la bibliothèque
standard, rien d'autre.

## 1. Trois lignes encore mystérieuses

Il reste exactement trois lignes du programme du chapitre 1 que nous n'avons
jamais ouvertes :

```python
loss = F.cross_entropy(model(x), y)                 # la loss qui valait 4.42 au départ
probas = F.softmax(scores / temperature, dim=-1)    # scores -> probabilités, avec une molette
i = torch.multinomial(probas, num_samples=1).item() # le tirage au sort du caractère suivant
```

Le fil rouge du chapitre : MiniLM vient de lire « La cigale et la fourm » et doit
prédire le caractère suivant. Trois candidats, « i », « o » et « u », et un score
brut par candidat, craché par la dernière couche du modèle. Ces trois nombres
nous accompagnent tout le notebook.

In [1]:
import math
import random

import torch
import torch.nn.functional as F

torch.manual_seed(42)                    # même hasard pour tous : résultats reproductibles

scores = torch.tensor([2.0, 1.0, 0.1])   # un score par candidat : i, o, u
print("scores bruts :", scores)

scores bruts : tensor([2.0000, 1.0000, 0.1000])


## 2. Des scores aux probabilités : le softmax

Une distribution de probabilités : des nombres **positifs ou nuls** dont la somme
vaut **exactement 1**. Première idée pour y arriver : diviser chaque score par la
somme des scores. Ça marche sur `[2.0, 1.0, 0.1]`... et voici le cas qui échoue,
dès qu'un score est négatif.

In [2]:
# Cas qui echoue : diviser chaque score par la somme casse des qu'un score est negatif.
scores_negatifs = [2.0, 1.0, -0.5]
total = sum(scores_negatifs)                       # 2.5
probas_naives = [s / total for s in scores_negatifs]
print("division naive :", probas_naives)           # [0.8, 0.4, -0.2] <- une « probabilite » negative !

division naive : [0.8, 0.4, -0.2]


### 2.3 Les deux gestes du softmax

La division naïve rate la positivité. La recette du softmax tient en deux gestes :

1. **exponentier** chaque score (`torch.exp`) : tout devient strictement positif,
   et l'ordre des scores est préservé ;
2. **diviser** chaque résultat par la somme de tous : le total vaut 1.

$$\mathrm{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

In [3]:
def softmax_maison(scores):
    """Transforme des scores bruts (logits) en distribution de probabilites."""
    exp_scores = torch.exp(scores)          # geste 1 : tout devient positif
    return exp_scores / exp_scores.sum()    # geste 2 : la somme vaut 1


probas = softmax_maison(scores)
print("probas    :", [round(p, 4) for p in probas.tolist()])   # 65.9 %, 24.2 %, 9.9 %
print("somme     :", round(probas.sum().item(), 6))            # 1.0 : une vraie distribution
print("F.softmax :", F.softmax(scores, dim=-1))                # identique : tu viens de le reecrire

probas    : [0.659, 0.2424, 0.0986]
somme     : 1.0
F.softmax : tensor([0.6590, 0.2424, 0.0986])


In [4]:
# Cas qui echoue : le softmax naif deborde sur de gros scores.
gros = torch.tensor([1000.0, 1000.0, 1000.0])
print("softmax naif  :", softmax_maison(gros))              # [nan, nan, nan] : exp(1000) = inf
# La parade : retrancher le maximum avant l'exponentielle (ne change rien au resultat).
print("softmax stable:", softmax_maison(gros - gros.max()))  # [1/3, 1/3, 1/3]

softmax naif  : tensor([nan, nan, nan])
softmax stable: tensor([0.3333, 0.3333, 0.3333])


### 2.4 La molette temperature, enfin démontée

La `temperature` du chapitre 1 : elle **divise les scores avant le softmax**, et
c'est tout. Diviser par $T < 1$ agrandit les écarts entre scores (distribution
piquée, texte sage), diviser par $T > 1$ les réduit (distribution aplatie, texte
créatif).

In [5]:
def softmax_temperature(scores, T):
    """Le softmax avec sa molette : divise les scores par T avant le softmax."""
    return softmax_maison(scores / T)


# Le tableau du chapitre, recalcule : T petit pique, T grand aplatit.
for nom, T in [("piquee ", 0.5), ("neutre ", 1.0), ("aplatie", 2.0)]:
    p = softmax_temperature(scores, T)
    print(f"T = {T} ({nom}) : " + "  ".join(f"{c} {v*100:4.1f} %" for c, v in zip("iou", p.tolist())))

T = 0.5 (piquee ) : i 86.4 %  o 11.7 %  u  1.9 %
T = 1.0 (neutre ) : i 65.9 %  o 24.2 %  u  9.9 %
T = 2.0 (aplatie) : i 50.2 %  o 30.4 %  u 19.4 %


## 3. Mesurer une prédiction : la cross-entropy

Pour juger le modèle, on regarde **la probabilité qu'il donnait à la bonne
réponse**, position après position, et on multiplie tout : c'est la
**vraisemblance**. Mais un produit de milliers de probabilités inférieures à 1
devient trop petit pour la machine (underflow). Le **logarithme** sauve tout :
il transforme le produit en somme, sans changer les poids qui gagnent.

In [10]:
# Cas qui echoue : le produit de 2 000 probabilites s'ecrase a zero machine.
p = 0.5
produit = 1.0
for _ in range(2000):        # seulement 2 000 caracteres, pas 38 331
    produit = produit * p
print("produit direct :", produit)             # 0.0 <- underflow

# La parade du logarithme : on additionne les logs au lieu de multiplier.
somme_logs = sum(math.log(p) for _ in range(2000))
print("somme des logs :", round(somme_logs, 1))    # environ -1386.3 : aucun debordement

# Sur les trois positions de la fable vues dans le chapitre :
logs = [math.log(0.9), math.log(0.8), math.log(0.2)]
print("-(log 0.9 + log 0.8 + log 0.2) =", round(-sum(logs), 4))      # 1.9379
print("divise par 3 exemples          =", round(-sum(logs) / 3, 4))  # 0.646 : une cross-entropy !

produit direct : 0.0
somme des logs : -1386.3
-(log 0.9 + log 0.8 + log 0.2) = 1.9379
divise par 3 exemples          = 0.646


La **cross-entropy** est la moyenne des $-\log$ des probabilités données aux
bonnes réponses :

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(\text{bonne réponse}_i)$$

En trois étapes sur un batch de logits : softmax ligne par ligne, lecture de la
probabilité de la bonne réponse dans chaque ligne, puis $-\log$ et moyenne.

In [11]:
def cross_entropy_maison(logits, cibles):
    """La formule du chapitre, mot a mot, sur un batch de logits."""
    probas = F.softmax(logits, dim=1)                # chaque ligne devient une distribution
    p_bonnes = probas[range(len(cibles)), cibles]    # la proba donnee a la bonne reponse, par ligne
    return -torch.log(p_bonnes).mean()               # -log, puis moyenne : la formule, mot a mot

In [17]:
# Le batch de trois exemples du chapitre : maison et PyTorch, memes chiffres.
logits = torch.tensor([[2.0, 1.0, 0.1],     # exemple 1 : la bonne reponse est i (indice 0)
                       [0.2, 2.3, 0.5],     # exemple 2 : la bonne reponse est o (indice 1)
                       [1.0, 1.0, 1.0]])    # exemple 3 : la bonne reponse est u (indice 2)
cibles = torch.tensor([0, 1, 2])            # les indices des bonnes reponses

print(f"cross-entropy maison : {cross_entropy_maison(logits, cibles).item():.4f}")   # 0.5895
print(f"F.cross_entropy      : {F.cross_entropy(logits, cibles).item():.4f}")        # identique : le softmax est deja dedans

cross-entropy maison : 0.5895
F.cross_entropy      : 0.5895


In [18]:
# La loss de 4.42 du chapitre 1, expliquee au centime.
print("modele parfait   (P = 1)    : loss =", abs(round(math.log(1.0), 4)))  # 0.0 pile
print("modele hesitant  (P = 0.5)  : loss =", round(-math.log(0.5), 4))      # 0.6931
print("confiant a cote  (P = 0.01) : loss =", round(-math.log(0.01), 4))     # 4.6052
print()
print("hasard pur sur 81 candidats : -log(1/81) =", round(math.log(81), 4))  # 4.3944, le 4.42 du ch. 1
print("loss finale 0.75 du ch. 1   : e^-0.75    =", round(math.exp(-0.75), 4),
      "-> 47 % a la bonne reponse en moyenne")

modele parfait   (P = 1)    : loss = 0.0
modele hesitant  (P = 0.5)  : loss = 0.6931
confiant a cote  (P = 0.01) : loss = 4.6052

hasard pur sur 81 candidats : -log(1/81) = 4.3944
loss finale 0.75 du ch. 1   : e^-0.75    = 0.4724 -> 47 % a la bonne reponse en moyenne


## 4. Écrire, c'est tirer au sort : l'échantillonnage

Dernier mystère : `torch.multinomial`. Tirer au sort « proportionnellement aux
probabilités », c'est découper le segment $[0, 1)$ en zones dont les tailles sont
les probabilités (les bornes sont les **sommes cumulées**), puis y laisser tomber
une fléchette uniforme : la zone où elle tombe désigne le gagnant.

In [19]:
def echantillonner(probas):
    """Tire un indice au sort, proportionnellement aux probabilites."""
    r = random.random()             # la flechette : uniforme entre 0 et 1
    cumul = 0.0
    for i, p in enumerate(probas):
        cumul += p                  # la borne droite de la zone du candidat i
        if r < cumul:               # la flechette est tombee dans cette zone
            return i
    return len(probas) - 1          # filet de securite (arrondis flottants)

In [20]:
# 10 000 tirages : le hasard obeit aux probabilites (66 % / 24 % / 10 % environ).
random.seed(0)                                     # flechettes reproductibles
tirages = [echantillonner([0.659, 0.242, 0.099]) for _ in range(10_000)]
for i, c in enumerate("iou"):
    n = tirages.count(i)
    print(f"« {c} » : {n:5d} tirages sur 10 000  ({n / 100:.1f} %)")

« i » :  6597 tirages sur 10 000  (66.0 %)
« o » :  2460 tirages sur 10 000  (24.6 %)
« u » :   943 tirages sur 10 000  (9.4 %)


In [21]:
# C'est exactement ce que fait torch.multinomial, en version vectorisee.
probas = softmax_maison(scores)                          # [0.659, 0.242, 0.099]
tirages = torch.multinomial(probas, num_samples=10_000, replacement=True)
comptes = torch.bincount(tirages, minlength=3)
for c, n in zip("iou", comptes.tolist()):
    print(f"« {c} » : {n:5d} tirages sur 10 000  ({n / 100:.1f} %)")

« i » :  6565 tirages sur 10 000  (65.7 %)
« o » :  2420 tirages sur 10 000  (24.2 %)
« u » :  1015 tirages sur 10 000  (10.2 %)


### 4.3 Relis generer : plus une seule ligne d'ombre

Remets bout à bout les trois sections du chapitre, et la fonction `generer` du
chapitre 1 se lit comme une phrase : le modèle calcule des scores, la température
les dose, le softmax en fait une distribution, le tirage désigne un caractère, le
contexte glisse d'un cran, et on recommence.

```python
def generer(prompt="\n", longueur=300, temperature=1.0):
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]
    sortie = []
    for _ in range(longueur):
        scores = model(torch.tensor([ctx]))              # les 81 logits (ch. 2, 6)
        probas = F.softmax(scores / temperature, dim=-1)  # scores -> distribution, dosée (§2)
        i = torch.multinomial(probas, num_samples=1).item()  # le tirage au sort (§4)
        sortie.append(itos[i])
        ctx = ctx[1:] + [i]                               # le gagnant rejoint le contexte
    return prompt + "".join(sortie)
```

Tu vas l'exécuter pour de vrai dans le bonus ci-dessous, sur le MiniLM du
chapitre 1 reconstruit sous tes yeux.

## 5. Bonus : les fonctions maison sur le MiniLM du chapitre 1

Le chapitre le promet : `F.cross_entropy` fait softmax, lecture de la bonne case,
log, signe moins et moyenne, en une seule ligne. Vérifions-le sur le vrai MiniLM,
pas sur un exemple jouet. On reconstruit le modèle du chapitre 1 (mêmes 91 497
nombres, même corpus de fables), on le réentraîne rapidement (2 000 étapes au
lieu de 8 000 : il ne s'agit pas de battre un record, juste d'avoir un modèle qui
a appris quelque chose), et à chaque étape clé, on calcule la loss **deux fois** :
avec `F.cross_entropy`, et avec la chaîne 100 % maison
`softmax_maison` + moins-log + moyenne. Mêmes chiffres, sinon rien.

### Le corpus du chapitre 1, embarqué

Trente fables de Jean de La Fontaine (domaine public, source Wikisource,
édition 1874), reprises telles quelles du notebook du chapitre 1 pour que ce
bonus tourne hors ligne.

In [22]:
corpus = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêter
Quelque grain pour subsister
Jusqu'à la saison nouvelle.
Je vous paierai, lui dit-elle,
Avant l'oût, foi d'animal,
Intérêt et principal.
La fourmi n'est pas prêteuse :
C'est là son moindre défaut.
Que faisiez-vous au temps chaud ?
Dit-elle à cette emprunteuse. -
Nuit et jour à tout venant
Je chantais, ne vous déplaise. -
Vous chantiez, j'en suis fort aise !
Eh bien ! dansez maintenant.


LE CORBEAU ET LE RENARD
Maître corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Hé ! bonjour, monsieur du corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le phénix des hôtes de ces bois.
À ces mots le corbeau ne se sent pas de joie ;
Et, pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le renard s'en saisit, et dit : Mon bon monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute :
Cette leçon vaut bien un fromage, sans doute.
Le corbeau, honteux et confus,
Jura, mais un peu tard, qu'on ne l'y prendrait plus.


LA GRENOUILLE QUI SE VEUT FAIRE AUSSI GROSSE QUE LE BŒUF
Une grenouille vit un bœuf
Qui lui sembla de belle taille.
Elle, qui n'était pas grosse en tout comme un œuf,
Envieuse, s'étend, et s'enfle, et se travaille
Pour égaler l'animal en grosseur ;
Disant : Regardez bien, ma sœur ;
Est-ce assez ? dites-moi ; n'y suis-je point encore ? -
Nenni. - M'y voici donc ? - Point du tout. - M'y voilà ? -
Vous n'en approchez point. La chétive pécore
S'enfla si bien qu'elle creva.
Le monde est plein de gens qui ne sont pas plus sages :
Tout bourgeois veut bâtir comme les grands seigneurs,
Tout petit prince a des ambassadeurs,
Tout marquis veut avoir des pages.


LE LOUP ET LE CHIEN
Un loup n'avait que les os et la peau,
Tant les chiens faisaient bonne garde.
Ce loup rencontre un dogue aussi puissant que beau,
Gras, poli, qui s'était fourvoyé par mégarde.
L'attaquer, le mettre en quartiers,
Sire loup l'eût fait volontiers :
Mais il fallait livrer bataille ;
Et le mâtin était de taille
À se défendre hardiment.
Le loup donc l'aborde humblement,
Entre en propos, et lui fait compliment
Sur son embonpoint, qu'il admire.
Il ne tiendra qu'à vous, beau sire,
D'être aussi gras que moi, lui repartit le chien.
Quittez les bois, vous ferez bien :
Vos pareils y sont misérables,
Cancres, hères et pauvres diables,
Dont la condition est de mourir de faim.
Car, quoi ! rien d'assuré ! point de franche lippée !
Tout à la pointe de l'épée !
Suivez-moi, vous aurez un bien meilleur destin.
Le loup reprit : Que me faudra-t-il faire ?
Presque rien, dit le chien : donner la chasse aux gens
Portants bâtons, et mendiants ;
Flatter ceux du logis, à son maître complaire ;
Moyennant quoi votre salaire
Sera force reliefs de toutes les façons,
Os de poulets, os de pigeons ;
Sans parler de mainte caresse.
Le loup déjà se forge une félicité
Qui le fait pleurer de tendresse.
Chemin faisant il vit le cou du chien pelé.
Qu'est-ce là ? lui dit-il. - Rien. - Quoi ! rien ! - Peu de chose. -
Mais encor ? - Le collier dont je suis attaché
De ce que vous voyez est peut-être la cause.
Attaché ! dit le loup : vous ne courez donc pas
Où vous voulez ? - Pas toujours ; mais qu'importe ?
Il importe si bien, que de tous vos repas
Je ne veux en aucune sorte,
Et ne voudrais pas même à ce prix d'un trésor.
Cela dit, maître loup s'enfuit, et court encor.


LA BESACE
Jupiter dit un jour : Que tout ce qui respire
S'en vienne comparaître aux pieds de ma grandeur :
Si dans son composé quelqu'un trouve à redire,
Il peut le déclarer sans peur ;
Je mettrai remède à la chose.
Venez, singe ; parlez le premier, et pour cause :
Voyez ces animaux, faites comparaison
De leurs beautés avec les vôtres.
Êtes-vous satisfait ? - Moi, dit-il ; pourquoi non ?
N'ai-je pas quatre pieds aussi bien que les autres ?
Mon portrait jusqu'ici ne m'a rien reproché :
Mais pour mon frère l'ours, on ne l'a qu'ébauché ;
Jamais, s'il me veut croire, il ne se fera peindre.
L'ours venant là-dessus, on crut qu'il s'allait plaindre.
Tant s'en faut : de sa forme il se loua très-fort ;
Glosa sur l'éléphant, dit qu'on pourrait encor
Ajouter à sa queue, ôter à ses oreilles ;
Que c'était une masse informe et sans beauté.
L'éléphant étant écouté,
Tout sage qu'il était, dit des choses pareilles :
Il jugea qu'à son appétit
Dame baleine était trop grosse.
Dame fourmi trouva le ciron trop petit,
Se croyant, pour elle, un colosse.
Jupin les renvoya s'étant censurés tous,
Du reste, contents d'eux. Mais, parmi les plus fous,
Notre espèce excella ; car, tout ce que nous sommes,
Lynx envers nos pareils, et taupes envers nous,
Nous nous pardonnons tout, et rien aux autres hommes :
On se voit d'un autre œil qu'on ne voit son prochain.
Le fabricateur souverain
Nous créa besaciers tous de même manière,
Tant ceux du temps passé que du temps d'aujourd'hui :
Il fit pour nos défauts la poche de derrière,
Et celle de devant pour les défauts d'autrui.


LE LOUP ET L'AGNEAU
La raison du plus fort est toujours la meilleure :
Nous l'allons montrer tout à l'heure.
Un agneau se désaltérait
Dans le courant d'une onde pure.
Un loup survint à jeun, qui cherchait aventure,
Et que la faim en ces lieux attirait.
Qui te rend si hardi de troubler mon breuvage ?
Dit cet animal plein de rage :
Tu seras châtié de ta témérité.
Sire, répond l'agneau, que Votre Majesté
Ne se mette pas en colère ;
Mais plutôt qu'elle considère
Que je me vas désaltérant
Dans le courant,
Plus de vingt pas au-dessous d'elle ;
Et que, par conséquent, en aucune façon
Je ne puis troubler sa boisson.
Tu la troubles ! reprit cette bête cruelle ;
Et je sais que de moi tu médis l'an passé.
Comment l'aurais-je fait, si je n'étais pas né ?
Reprit l'agneau : je tette encore ma mère. -
Si ce n'est toi, c'est donc ton frère. -
Je n'en ai point. - C'est donc quelqu'un des tiens ;
Car vous ne m'épargnez guère,
Vous, vos bergers et vos chiens.
On me l'a dit : il faut que je me venge.
Là-dessus, au fond des forêts
Le loup l'emporte, et puis le mange,
Sans autre forme de procès.


LA MORT ET LE BÛCHERON
Un pauvre bucheron, tout couvert de ramée,
Sous le faix du fagot aussi bien que des ans,
Gémissant et courbé, marchait à pas pesants,
Et tâchait de gagner sa chaumine enfumée.
Enfin, n'en pouvant plus d'effort et de douleur,
Il met bas son fagot, il songe à son malheur.
Quel plaisir a-t-il eu depuis qu'il est au monde ?
En est-il un plus pauvre en la machine ronde ?
Point de pain quelquefois, et jamais de repos :
Sa femme, ses enfants, les soldats, les impôts,
Le créancier, et la corvée,
Lui font d'un malheureux la peinture achevée.
Il appelle la Mort. Elle vient sans tarder,
Lui demande ce qu'il faut faire.
C'est, dit-il, afin de m'aider
À recharger ce bois ; tu ne tarderas guère.
Le trépas vient tout guérir ;
Mais ne bougeons d'où nous sommes :
Plutôt souffrir que mourir,
C'est la devise des hommes.


LE RENARD ET LA CIGOGNE
Compère le renard se mit un jour en frais,
Et retint à dîner commère la cigogne.
Le régal fut petit et sans beaucoup d'apprêts :
Le galant, pour toute besogne,
Avait un brouet clair ; il vivait chichement.
Ce brouet fut par lui servi sur une assiette :
La cigogne au long bec n'en put attraper miette ;
Et le drôle eut lapé le tout en un moment.
Pour se venger de cette tromperie,
À quelque temps de là la cigogne le prie.
Volontiers, lui dit-il ; car avec mes amis
Je ne fais point cérémonie.
À l'heure dite, il courut au logis
De la cigogne son hôtesse ;
Loua très-fort sa politesse ;
Trouva le dîner cuit à point :
Bon appétit surtout ; renards n'en manquent point.
Il se réjouissait à l'odeur de la viande
Mise en menus morceaux, et qu'il croyait friande.
On servit, pour l'embarrasser,
En un vase à long col et d'étroite embouchure.
Le bec de la cigogne y pouvait bien passer ;
Mais le museau du sire était d'autre mesure.
Il lui fallut à jeun retourner au logis,
Honteux comme un renard qu'une poule aurait pris,
Serrant la queue, et portant bas l'oreille.
Trompeurs, c'est pour vous que j'écris :
Attendez-vous à la pareille.


LE CHÊNE ET LE ROSEAU
Le chêne un jour dit au roseau :
Vous avez bien sujet d'accuser la nature ;
Un roitelet pour vous est un pesant fardeau :
Le moindre vent qui d'aventure
Fait rider la face de l'eau,
Vous oblige à baisser la tête ;
Cependant que mon front, au Caucase pareil,
Non content d'arrêter les rayons du soleil,
Brave l'effort de la tempête.
Tout vous est aquilon, tout me semble zéphyr.
Encor si vous naissiez à l'abri du feuillage
Dont je couvre le voisinage,
Vous n'auriez pas tant à souffrir,
Je vous défendrais de l'orage :
Mais vous naissez le plus souvent
Sur les humides bords des royaumes du vent.
La nature envers vous me semble bien injuste.
Votre compassion, lui répondit l'arbuste,
Part d'un bon naturel ; mais quittez ce souci :
Les vents me sont moins qu'à vous redoutables ;
Je plie et ne romps pas. Vous avez jusqu'ici
Contre leurs coups épouvantables
Résisté sans courber le dos ;
Mais attendons la fin. Comme il disait ces mots,
Du bout de l'horizon accourt avec furie
Le plus terrible des enfants
Que le Nord eût portés jusque-là dans ses flancs.
L'arbre tient bon ; le roseau plie.
Le vent redouble ses efforts,
Et fait si bien qu'il déracine
Celui de qui la tête au ciel était voisine,
Et dont les pieds touchaient à l'empire des morts.


LE LION ET LE RAT
Il faut, autant qu'on peut, obliger tout le monde :
On a souvent besoin d'un plus petit que soi.
De cette vérité deux fables feront foi ;
Tant la chose en preuves abonde.
Entre les pattes d'un lion
Un rat sortit de terre assez à l'étourdie.
Le roi des animaux, en cette occasion,
Montra ce qu'il était, et lui donna la vie.
Ce bienfait ne fut pas perdu.
Quelqu'un aurait-il jamais cru
Qu'un lion d'un rat eût affaire ?
Cependant il advint qu'au sortir des forêts
Ce lion fut pris dans des rets,
Dont ses rugissements ne le purent défaire.
Sire rat accourut, et fit tant par ses dents
Qu'une maille rongée emporta tout l'ouvrage.
Patience et longueur de temps
Font plus que force ni que rage.


LA COLOMBE ET LA FOURMI
L'autre exemple est tiré d'animaux plus petits.
Le long d'un clair ruisseau buvait une colombe,
Quand sur l'eau se penchant une fourmis y tombe ;
Et dans cet océan on eût vu la fourmis
S'efforcer, mais en vain, de regagner la rive.
La colombe aussitôt usa de charité :
Un brin d'herbe dans l'eau par elle étant jeté,
Ce fut un promontoire où la fourmis arrive.
Elle se sauve. Et là-dessus
Passe un certain croquant qui marchait les pieds nus :
Ce croquant, par hasard, avait une arbalète.
Dès qu'il voit l'oiseau de Vénus,
Il le croit en son pot, et déjà lui fait fête.
Tandis qu'à le tuer mon villageois s'apprête,
La fourmi le pique au talon.
Le vilain retourne la tête :
La colombe l'entend, part, et tire de long.
Le souper du croquant avec elle s'envole :
Point de pigeon pour une obole.


LE LIÈVRE ET LA TORTUE
Rien ne sert de courir ; il faut partir à point :
Le lièvre et la tortue en sont un témoignage.
Gageons, dit celle-ci, que vous n'atteindrez point
Sitôt que moi ce but. Sitôt ! êtes-vous sage ?
Repartit l'animal léger :
Ma commère, il vous faut purger
Avec quatre grains d'ellébore.
- Sage ou non, je parie encore.
Ainsi fut fait ; et de tous deux
On mit près du but les enjeux.
Savoir quoi, ce n'est pas l'affaire,
Ni de quel juge l'on convint.
Notre lièvre n'avait que quatre pas à faire ;
J'entends de ceux qu'il fait lorsque, près d'être atteint,
Il s'éloigne des chiens, les renvoie aux calendes,
Et leur fait arpenter les landes.
Ayant, dis-je, du temps de reste pour brouter,
Pour dormir, et pour écouter
D'où vient le vent, il laisse la tortue
Aller son train de sénateur.
Elle part, elle s'évertue ;
Elle se hâte avec lenteur.
Lui cependant méprise une telle victoire,
Tient la gageure à peu de gloire,
Croit qu'il y va de son honneur
De partir tard. Il broute, il se repose ;
Il s'amuse à toute autre chose
Qu'à la gageure. À la fin, quand il vit
Que l'autre touchait presque au bout de la carrière,
Il partit comme un trait ; mais les élans qu'il fit
Furent vains : la tortue arriva la première.
Eh bien ! lui cria-t-elle, avais-je pas raison ?
De quoi vous sert votre vitesse ?
Moi l'emporter ! et que serait-ce
Si vous portiez une maison ?


LE RENARD ET LES RAISINS
Certain renard gascon, d'autres disent normand,
Mourant presque de faim, vit au haut d'une treille
Des raisins, mûrs apparemment,
Et couverts d'une peau vermeille.
Le galant en eût fait volontiers un repas ;
Mais comme il n'y pouvait atteindre :
Ils sont trop verts, dit-il, et bons pour des goujats.
Fit-il pas mieux que de se plaindre ?


LE HÉRON
Un jour, sur ses longs pieds, allait je ne sais où
Le héron au long bec emmanché d'un long cou :
Il côtoyait une rivière.
L'onde était transparente ainsi qu'aux plus beaux jours ;
Ma commère la carpe y faisait mille tours
Avec le brochet son compère.
Le héron en eût fait aisément son profit :
Tous approchaient du bord ; l'oiseau n'avait qu'à prendre.
Mais il crut mieux faire d'attendre
Qu'il eût un peu plus d'appétit :
Il vivait de régime, et mangeait à ses heures.
Après quelques moments l'appétit vint : l'oiseau,
S'approchant du bord, vit sur l'eau
Des tanches qui sortaient du fond de ces demeures.
Le mets ne lui plut pas, il s'attendait à mieux,
Et montrait un goût dédaigneux
Comme le rat du bon Horace.
Moi, des tanches ! dit-il ; moi, héron, que je fasse
Une si pauvre chère ! Et pour qui me prend-on ?
La tanche rebutée, il trouva du goujon.
Du goujon ! c'est bien là le dîner d'un héron !
J'ouvrirais pour si peu le bec ! aux dieux ne plaise !
Il l'ouvrit pour bien moins : tout alla de façon
Qu'il ne vit plus aucun poisson.
La faim le prit : il fut tout heureux et tout aise
De rencontrer un limaçon.
Ne soyons pas si difficiles :
Les plus accommodants, ce sont les plus habiles ;
On hasarde de perdre en voulant trop gagner.
Gardez-vous de rien dédaigner,
Surtout quand vous avez à peu près votre compte.
Bien des gens y sont pris. Ce n'est pas aux hérons
Que je parle : écoutez, humains, un autre conte :
Vous verrez que chez vous j'ai puisé ces leçons.


LA LAITIÈRE ET LE POT AU LAIT
Perrette, sur sa tête ayant un pot au lait
Bien posé sur un coussinet,
Prétendait arriver sans encombre à la ville.
Légère et court vêtue, elle allait à grands pas,
Ayant mis ce jour-là, pour être plus agile,
Cotillon simple et souliers plats.
Notre laitière ainsi troussée
Comptait déjà dans sa pensée
Tout le prix de son lait ; en employait l'argent ;
Achetait un cent d'œufs ; faisait triple couvée :
La chose allait à bien par son soin diligent.
Il m'est, disait-elle, facile
D'élever des poulets autour de ma maison ;
Le renard sera bien habile
S'il ne m'en laisse assez pour avoir un cochon.
Le porc à s'engraisser coûtera peu de son ;
Il était, quand je l'eus, de grosseur raisonnable :
J'aurai, le revendant, de l'argent bel et bon.
Et qui m'empêchera de mettre en notre étable,
Vu le prix dont il est, une vache et son veau,
Que je verrai sauter au milieu du troupeau ?
Perrette là-dessus saute aussi, transportée :
Le lait tombe ; adieu veau, vache, cochon, couvée.
La dame de ces biens, quittant d'un œil marri
Sa fortune ainsi répandue,
Va s'excuser à son mari,
En grand danger d'être battue.
Le récit en farce en fut fait ;
On l'appela le Pot au lait.
Quel esprit ne bat la campagne ?
Qui ne fait châteaux en Espagne ?
Picrochole, Pyrrhus, la laitière, enfin tous,
Autant les sages que les fous.
Chacun songe en veillant ; il n'est rien de plus doux
Une flatteuse erreur emporte alors nos âmes ;
Tout le bien du monde est à nous,
Tous les honneurs, toutes les femmes.
Quand je suis seul, je fais au plus brave un défi ;
Je m'écarte, je vais détrôner le sophi ;
On m'élit roi, mon peuple m'aime ;
Les diadèmes vont sur ma tête pleuvant :
Quelque accident fait-il que je rentre en moi-même ;
Je suis Gros-Jean comme devant.


LE COCHE ET LA MOUCHE
Dans un chemin montant, sablonneux, malaisé,
Et de tous les côtés au soleil exposé,
Six forts chevaux tiraient un coche.
Femmes, moine, vieillards, tout était descendu :
L'attelage suait, soufflait, était rendu.
Une mouche survient, et des chevaux s'approche,
Prétend les animer par son bourdonnement,
Pique l'un, pique l'autre, et pense à tout moment
Qu'elle fait aller la machine,
S'assied sur le timon, sur le nez du cocher.
Aussitôt que le char chemine,
Et qu'elle voit les gens marcher,
Elle s'en attribue uniquement la gloire,
Va, vient, fait l'empressée : il semble que ce soit
Un sergent de bataille allant en chaque endroit
Faire avancer ses gens et hâter la victoire.
La mouche, en ce commun besoin,
Se plaint qu'elle agit seule, et qu'elle a tout le soin ;
Qu'aucun n'aide aux chevaux à se tirer d'affaire.
Le moine disait son bréviaire :
Il prenait bien son temps ! une femme chantait :
C'était bien de chansons qu'alors il s'agissait !
Dame mouche s'en va chanter à leurs oreilles,
Et fait cent sottises pareilles.
Après bien du travail, le coche arrive au haut.
Respirons maintenant ! dit la mouche aussitôt :
J'ai tant fait que nos gens sont enfin dans la plaine.
Çà, messieurs les chevaux, payez-moi de ma peine.
Ainsi certaines gens, faisant les empressés,
S'introduisent dans les affaires :
Ils font partout les nécessaires,
Et, partout importuns, devraient être chassés.


LE SAVETIER ET LE FINANCIER
Un savetier chantait du matin jusqu'au soir :
C'était merveille de le voir,
Merveille de l'ouïr ; il faisait des passages :
Plus content qu'aucun des sept sages.
Son voisin, au contraire, étant tout cousu d'or,
Chantait peu, dormait moins encor :
C'était un homme de finance.
Si sur le point du jour parfois il sommeillait,
Le savetier alors en chantant l'éveillait ;
Et le financier se plaignait
Que les soins de la Providence
N'eussent pas au marché fait vendre le dormir,
Comme le manger et le boire.
En son hôtel il fait venir
Le chanteur, et lui dit : Or çà, sire Grégoire,
Que gagnez-vous par an ? Par an ! ma foi, monsieur
Dit avec un ton de rieur
Le gaillard savetier, ce n'est point ma manière
De compter de la sorte ; et je n'entasse guère
Un jour sur l'autre : il suffit qu'à la fin
J'attrape le bout de l'année ;
Chaque jour amène son pain. -
Eh bien ! que gagnez-vous, dites-moi, par journée ?
Tantôt plus, tantôt moins : le mal est que toujours
(Et sans cela nos gains seraient assez honnêtes),
Le mal est que dans l'an s'entremêlent des jours
Qu'il faut chômer ; on nous ruine en fêtes :
L'une fait tort à l'autre ; et monsieur le curé
De quelque nouveau saint charge toujours son prône.
Le financier, riant de sa naïveté,
Lui dit : Je vous veux mettre aujourd'hui sur le trône.
Prenez ces cent écus ; gardez-les avec soin,
Pour vous en servir au besoin.
Le savetier crut voir tout l'argent que la terre
Avait depuis plus de cent ans,
Produit pour l'usage des gens.
Il retourne chez lui : dans sa cave il enserre
L'argent, et sa joie à la fois.
Plus de chant : il perdit la voix
Du moment qu'il gagna ce qui cause nos peines.
Le sommeil quitta son logis :
Il eut pour hôtes les soucis,
Les soupçons, les alarmes vaines.
Tout le jour il avait l'œil au guet ; et la nuit,
Si quelque chat faisait du bruit,
Le chat prenait l'argent. À la fin le pauvre homme
S'en courut chez celui qu'il ne réveillait plus :
Rendez-moi, lui dit-il, mes chansons et mon somme ;
Et reprenez vos cent écus.


LES ANIMAUX MALADES DE LA PESTE
Un mal qui répand la terreur,
Mal que le ciel en sa fureur
Inventa pour punir les crimes de la terre,
La peste (puisqu'il faut l'appeler par son nom),
Capable d'enrichir en un jour l'Achéron,
Faisait aux animaux la guerre.
Ils ne mouraient pas tous, mais tous étaient frappés :
On n'en voyait point d'occupés
À chercher le soutien d'une mourante vie ;
Nul mets n'excitait leur envie ;
Ni loups ni renards n'épiaient
La douce et l'innocente proie ;
Les tourterelles se fuyaient :
Plus d'amour, partant plus de joie.
Le lion tint conseil, et dit : Mes chers amis,
Je crois que le ciel a permis
Pour nos péchés cette infortune.
Que le plus coupable de nous
Se sacrifie aux traits du céleste courroux ;
Peut-être il obtiendra la guérison commune.
L'histoire nous apprend qu'en de tels accidents
On fait de pareils dévouements.
Ne nous flattons donc point ; voyons sans indulgence
L'état de notre conscience.
Pour moi, satisfaisant mes appétits gloutons,
J'ai dévoré force moutons.
Que m'avaient-ils fait ? nulle offense ;
Même il m'est arrivé quelquefois de manger
Le berger.
Je me dévouerai donc, s'il le faut : mais je pense
Qu'il est bon que chacun s'accuse ainsi que moi ;
Car on doit souhaiter, selon toute justice,
Que le plus coupable périsse.
Sire, dit le renard, vous êtes trop bon roi ;
Vos scrupules font voir trop de délicatesse.
Eh bien ! manger moutons, canaille, sotte espèce,
Est-ce un péché ? Non, non. Vous leur fîtes, seigneur,
En les croquant, beaucoup d'honneur ;
Et quant au berger, l'on peut dire
Qu'il était digne de tous maux,
Étant de ces gens-là qui sur les animaux
Se font un chimérique empire.
Ainsi dit le renard ; et flatteurs d'applaudir.
On n'osa trop approfondir
Du tigre, ni de l'ours, ni des autres puissances,
Les moins pardonnables offenses :
Tous les gens querelleurs, jusqu'aux simples mâtins,
Au dire de chacun, étaient de petits saints.
L'âne vint à son tour, et dit : J'ai souvenance
Qu'en un pré de moines passant,
La faim, l'occasion, l'herbe tendre, et, je pense,
Quelque diable aussi me poussant,
Je tondis de ce pré la largeur de ma langue ;
Je n'en avais nul droit, puisqu'il faut parler net.
À ces mots, on cria haro sur le baudet.
Un loup, quelque peu clerc, prouva par sa harangue
Qu'il fallait dévouer ce maudit animal,
Ce pelé, ce galeux, d'où venait tout leur mal.
Sa peccadille fut jugée un cas pendable.
Manger l'herbe d'autrui ! quel crime abominable !
Rien que la mort n'était capable
D'expier son forfait. On le lui fit bien voir.
Selon que vous serez puissant ou misérable,
Les jugements de cour vous rendront blanc ou noir.


LA POULE AUX ŒUFS D'OR
L'avarice perd tout en voulant tout gagner.
Je ne veux, pour le témoigner,
Que celui dont la poule, à ce que dit la fable,
Pondait tous les jours un œuf d'or.
Il crut que, dans son corps, elle avait un trésor ;
Il la tua, l'ouvrit, et la trouva semblable
À celle dont les œufs ne lui rapportaient rien,
S'étant lui-même ôté le plus beau de son bien.
Belle leçon pour les gens chiches !
Pendant ces derniers temps combien en a-t-on vus
Qui du soir au matin sont pauvres devenus
Pour vouloir trop tôt être riches !


L'OURS ET LES DEUX COMPAGNONS
Deux compagnons, pressés d'argent,
À leur voisin fourreur vendirent
La peau d'un ours encor vivant,
Mais qu'ils tueraient bientôt, du moins à ce qu'ils dirent,
C'était le roi des ours au compte de ces gens.
Le marchand à sa peau devait faire fortune ;
Elle garantirait des froids les plus cuisants ;
On en pourrait fourrer plutôt deux robes qu'une.
Dindenaut prisait moins ses moutons qu'eux leur ours :
Leur, à leur compte, et non à celui de la bête.
S'offrant de la livrer au plus tard dans deux jours,
Ils conviennent de prix, et se mettent en quête,
Trouvent l'ours qui s'avance et vient vers eux au trot,
Voilà mes gens frappés comme d'un coup de foudre.
Le marché ne tint pas ; il fallut le résoudre :
D'intérêts contre l'ours, on n'en dit pas un mot.
L'un des deux compagnons grimpe au faîte d'un arbre ;
L'autre, plus froid que n'est un marbre,
Se couche sur le nez, fait le mort, tient son vent,
Ayant quelque part ouï dire
Que l'ours s'acharne peu souvent
Sur un corps qui ne vit, ne meut, ni ne respire.
Seigneur ours, comme un sot, donna dans ce panneau :
Il voit ce corps gisant, le croit privé de vie ;
Et, de peur de supercherie,
Le tourne, le retourne, approche son museau,
Flaire aux passages de l'haleine.
C'est, dit-il, un cadavre ; ôtons-nous, car il sent.
À ces mots, l'ours s'en va dans la forêt prochaine.
L'un de nos deux marchands de son arbre descend,
Court à son compagnon, lui dit que c'est merveille
Qu'il n'ait eu seulement que la peur pour tout mal.
Eh bien ! ajouta-t-il, la peau de l'animal ?
Mais que t'a-t-il dit à l'oreille ?
Car il t'approchait de bien près,
Te retournant avec sa serre.
Il m'a dit qu'il ne faut jamais
Vendre la peau de l'ours qu'on ne l'ait mis par terre.


LE RENARD ET LE BOUC
Capitaine renard allait de compagnie
Avec son ami bouc des plus haut encornés :
Celui-ci ne voyait pas plus loin que son nez ;
L'autre était passé maître en fait de tromperie.
La soif les obligea de descendre en un puits ;
Là chacun d'eux se désaltère.
Après qu'abondamment tous deux en eurent pris,
Le renard dit au bouc : Que ferons-nous, compère ?
Ce n'est pas tout de boire, il faut sortir d'ici.
Lève tes pieds en haut, et tes cornes aussi ;
Mets-les contre le mur : le long de ton échine
Je grimperai premièrement ;
Puis sur tes cornes m'élevant,
À l'aide de cette machine,
De ce lieu-ci je sortirai,
Après quoi je t'en tirerai.
Par ma barbe, dit l'autre, il est bon ; et je loue
Les gens bien sensés comme toi.
Je n'aurais jamais, quant à moi,
Trouvé ce secret, je l'avoue.
Le renard sort du puits, laisse son compagnon,
Et vous lui fait un beau sermon
Pour l'exhorter à patience.
Si le ciel t'eût, dit-il, donné par excellence
Autant de jugement que de barbe au menton,
Tu n'aurais pas, à la légère,
Descendu dans ce puits. Or, adieu ; j'en suis hors :
Tâche de t'en tirer et fais tous les efforts ;
Car, pour moi, j'ai certaine affaire
Qui ne me permet pas d'arrêter en chemin.
En toute chose il faut considérer la fin.


LE CERF SE VOYANT DANS L'EAU
Dans le cristal d'une fontaine
Un cerf se mirant autrefois
Louait la beauté de son bois,
Et ne pouvait qu'avecque peine
Souffrir ses jambes de fuseaux,
Dont il voyait l'objet se perdre dans les eaux.
Quelle proportion de mes pieds à ma tête !
Disait-il en voyant leur ombre avec douleur :
Des taillis les plus hauts mon front atteint le faîte ;
Mes pieds ne me font point d'honneur.
Tout en parlant de la sorte,
Un limier le fait partir.
Il tâche à se garantir ;
Dans les forêts il s'emporte :
Son bois, dommageable ornement,
L'arrêtant à chaque moment,
Nuit à l'office que lui rendent
Ses pieds de qui ses jours dépendent.
Il se dédit alors, et maudit les présents
Que le ciel lui fait tous les ans.
Nous faisons cas du beau, nous méprisons l'utile ;
Et le beau souvent nous détruit.
Ce cerf blâme ses pieds qui le rendent agile ;
Il estime un bois qui lui nuit.


LE LOUP DEVENU BERGER
Un loup qui commençait d'avoir petite part
Aux brebis de son voisinage,
Crut qu'il fallait s'aider de la peau du renard,
Et faire un nouveau personnage
Il s'habille en berger, endosse un hoqueton,
Fait sa houlette d'un bâton,
Sans oublier la cornemuse.
Pour pousser jusqu'au bout la ruse,
Il aurait volontiers écrit sur son chapeau :
"C'est moi qui suis Guillot, berger de ce troupeau."
Sa personne étant ainsi faite,
Et ses pieds de devant posés sur sa houlette,
Guillot le sycophante approche doucement.
Guillot, le vrai Guillot, étendu sur l'herbette,
Dormait alors profondément ;
Son chien dormait aussi, comme aussi sa musette :
La plupart des brebis dormaient pareillement.
L'hypocrite les laissa faire ;
Et, pour pouvoir mener vers son fort les brebis,
Il voulut ajouter la parole aux habits,
Chose qu'il croyait nécessaire ;
Mais cela gâta son affaire :
Il ne put du pasteur contrefaire la voix.
Le ton dont il parla fit retentir les bois,
Et découvrit tout le mystère.
Chacun se réveille à ce son,
Les brebis, le chien, le garçon.
Le pauvre loup, dans cet esclandre,
Empêché par son hoqueton,
Ne put ni fuir ni se défendre.
Toujours par quelque endroit fourbes se laissent prendre
Quiconque est loup agisse en loup ;
C'est le plus certain de beaucoup.


LE RAT DE VILLE, ET LE RAT DES CHAMPS
Autrefois le rat de ville
Invita le rat des champs,
D'une façon fort civile,
À des reliefs d'ortolans.
Sur un tapis de Turquie
Le couvert se trouva mis.
Je laisse à penser la vie
Que firent ces deux amis.
Le régal fut fort honnête ;
Rien ne manquait au festin :
Mais quelqu'un troubla la fête
Pendant qu'ils étaient en train.
À la porte de la salle
Ils entendirent du bruit :
Le rat de ville détale ;
Son camarade le suit.
Le bruit cesse, on se retire :
Rats en campagne aussitôt ;
Et le citadin de dire :
Achevons tout notre rôt.
C'est assez, dit le rustique :
Demain vous viendrez chez moi.
Ce n'est pas que je me pique
De tous vos festins de roi :
Mais rien ne vient m'interrompre ;
Je mange tout à loisir.
Adieu donc. Fi du plaisir
Que la crainte peut corrompre !


LE PETIT POISSON ET LE PÊCHEUR
Petit poisson deviendra grand,
Pourvu que Dieu lui prête vie ;
Mais le lâcher en attendant,
Je tiens pour moi que c'est folie,
Car de le rattraper il n'est pas trop certain.
Un carpeau qui n'était encore que fretin,
Fut pris par un pêcheur au bord d'une rivière.
Tout fait nombre, dit l'homme, en voyant son butin ;
Voilà commencement de chère et de festin :
Mettons-le en notre gibecière.
Le pauvre carpillon lui dit en sa manière :
Que ferez-vous de moi ? je ne saurais fournir
Au plus qu'une demi-bouchée.
Laissez-moi carpe devenir :
Je serai par vous repêchée ;
Quelque gros partisan m'achètera bien cher :
Au lieu qu'il vous en faut chercher
Peut-être encor cent de ma taille
Pour faire un plat : quel plat ! croyez-moi, rien qui vaille.
Rien qui vaille ! eh bien ! soit, repartit le pêcheur ;
Poisson, mon bel ami, qui faites le prêcheur,
Vous irez dans la poêle ; et, vous avez beau dire,
Dès ce soir on vous fera frire.
Un Tiens, vaut, ce dit-on, mieux que deux Tu l'auras :
L'un est sûr, l'autre ne l'est pas.


LE POT DE TERRE ET LE POT DE FER
Le pot de fer proposa
Au pot de terre un voyage.
Celui-ci s'en excusa,
Disant qu'il ferait que sage
De garder le coin du feu :
Car il lui fallait si peu,
Si peu que la moindre chose
De son débris serait cause :
Il n'en reviendrait morceau.
Pour vous, dit-il, dont la peau
Est plus dure que la mienne,
Je ne vois rien qui vous tienne.
Nous vous mettrons à couvert,
Repartit le pot de fer :
Si quelque matière dure
Vous menace d'aventure,
Entre deux je passerai,
Et du coup vous sauverai.
Cette offre le persuade.
Pot de fer son camarade
Se met droit à ses côtés.
Mes gens s'en vont à trois pieds
Clopin clopant comme ils peuvent,
L'un contre l'autre jetés
Au moindre hoquet qu'ils treuvent.
Le pot de terre en souffre ; il n'eut pas fait cent pas
Que par son compagnon il fut mis en éclats,
Sans qu'il eût lieu de se plaindre.
Ne nous associons qu'avecque nos égaux ;
Ou bien il nous faudra craindre
Le destin d'un de ces pots.


LE LABOUREUR ET SES ENFANTS
Travaillez, prenez de la peine :
C'est le fonds qui manque le moins.
Un riche laboureur, sentant sa mort prochaine,
Fit venir ses enfants, leur parla sans témoins.
Gardez-vous, leur dit-il, de vendre l'héritage
Que nous ont laissé nos parents :
Un trésor est caché dedans.
Je ne sais pas l'endroit ; mais un peu de courage
Vous le fera trouver : vous en viendrez à bout.
Remuez votre champ dès qu'on aura fait l'oût :
Creusez, fouillez, bêchez ; ne laissez nulle place
Où la main ne passe et repasse.
Le père mort, les fils vous retournent le champ,
De çà, de là, partout ; si bien qu'au bout de l'an
Il en rapporta davantage.
D'argent, point de caché. Mais le père fut sage
De leur montrer, avant sa mort,
Que le travail est un trésor.


LE MEUNIER, SON FILS, ET L'ÂNE
L'invention des arts étant un droit d'aînesse,
Nous devons l'apologue à l'ancienne Grèce :
Mais ce champ ne se peut tellement moissonner
Que les derniers venus n'y trouvent à glaner.
La feinte est un pays plein de terres désertes ;
Tous les jours nos auteurs y font des découvertes.
Je t'en veux dire un trait assez bien inventé :
Autrefois à Racan Malherbe l'a conté.
Ces deux rivaux d'Horace, héritiers de sa lyre,
Disciples d'Apollon, nos maîtres, pour mieux dire,
Se rencontrant un jour tout seuls et sans témoins
(Comme ils se confiaient leurs pensers et leurs soins),
Racan commence ainsi : Dites-moi, je vous prie,
Vous qui devez savoir les choses de la vie,
Qui par tous ses degrés avez déjà passé,
Et que rien ne doit fuir en cet âge avancé,
À quoi me résoudrai-je ? Il est temps que j'y pense.
Vous connaissez mon bien, mon talent, ma naissance :
Dois-je dans la province établir mon séjour,
Prendre emploi dans l'armée, ou bien charge à la cour ?
Tout au monde est mêlé d'amertume et de charmes :
La guerre a ses douceurs, l'hymen a ses alarmes.
Si je suivais mon goût, je saurais où buter ;
Mais j'ai les miens, la cour, le peuple à contenter.
Malherbe là-dessus : Contenter tout le monde !
Écoutez ce récit avant que je réponde.
J'ai lu dans quelque endroit qu'un meunier et son fils,
L'un vieillard, l'autre enfant, non pas des plus petits,
Mais garçon de quinze ans, si j'ai bonne mémoire,
Allaient vendre leur âne, un certain jour de foire.
Afin qu'il fût plus frais et de meilleur débit,
On lui lia les pieds, on vous le suspendit ;
Puis cet homme et son fils le portent comme un lustre.
Pauvres gens ! idiots ! couple ignorant et rustre !
Le premier qui les vit de rire s'éclata :
Quelle farce, dit-il, vont jouer ces gens-là ?
Le plus âne des trois n'est pas celui qu'on pense.
Le meunier, à ces mots, connaît son ignorance ;
Il met sur pieds sa bête, et la fait détaler.
L'âne, qui goûtait fort l'autre façon d'aller,
Se plaint en son patois. Le meunier n'en a cure,
Il fait monter son fils, il suit : et, d'aventure,
Passent trois bons marchands. Cet objet leur déplut.
Le plus vieux au garçon s'écria tant qu'il put :
Oh là ! oh ! descendez, que l'on ne vous le dise,
Jeune homme, qui menez laquais à barbe grise !
C'était à vous de suivre, au vieillard de monter.
Messieurs, dit le meunier, il vous faut contenter.
L'enfant met pied à terre, et puis le vieillard monte ;
Quand trois filles passant, l'une dit : C'est grand'honte
Qu'il faille voir ainsi clocher ce jeune fils,
Tandis que ce nigaud, comme un évêque assis,
Fait le veau sur son âne, et pense être bien sage.
Il n'est, dit le meunier, plus de veaux à mon âge :
Passez votre chemin, la fille, et m'en croyez.
Après maints quolibets coup sur coup renvoyés,
L'homme crut avoir tort, et mit son fils en croupe.
Au bout de trente pas, une troisième troupe
Trouve encore à gloser. L'un dit : Ces gens sont fous !
Le baudet n'en peut plus ; il mourra sous leurs coups.
Eh quoi ! charger ainsi cette pauvre bourrique !
N'ont-ils point de pitié de leur vieux domestique ?
Sans doute qu'à la foire ils vont vendre sa peau.
Parbleu ! dit le meunier, est bien fou du cerveau
Qui prétend contenter tout le monde et son père.
Essayons toutefois si par quelque manière
Nous en viendrons à bout. Ils descendent tous deux.
L'âne se prélassant marche seul devant eux.
Un quidam les rencontre et dit : Est-ce la mode
Que baudet aille à l'aise, et meunier s'incommode ?
Qui de l'âne ou du maître est fait pour se lasser ?
Je conseille à ces gens de le faire enchâsser.
Ils usent leurs souliers, et conservent leur âne !
Nicolas, au rebours : car, quand il va voir Jeanne,
Il monte sur sa bête ; et la chanson le dit.
Beau trio de baudets ! le meunier repartit :
Je suis âne, il est vrai, j'en conviens, je l'avoue ;
Mais que dorénavant on me blâme, on me loue,
Qu'on dise quelque chose ou qu'on ne dise rien,
J'en veux faire à ma tête. Il le fit, et fit bien.
Quant à vous, suivez Mars, ou l'Amour, ou le prince ;
Allez, venez, courez ; demeurez en province ;
Prenez femme, abbaye, emploi, gouvernement :
Les gens en parleront, n'en doutez nullement.


LE CHAT, LA BELETTE, ET LE PETIT LAPIN
Du palais d'un jeune lapin
Dame belette, un beau matin,
S'empara : c'est une rusée.
Le maître étant absent, ce lui fut chose aisée.
Elle porta chez lui ses pénates, un jour
Qu'il était allé faire à l'Aurore sa cour
Parmi le thym et la rosée.
Après qu'il eut brouté, trotté, fait tous ses tours,
Jeannot lapin retourne aux souterrains séjours.
La belette avait mis le nez à la fenêtre.
Ô dieux hospitaliers ! que vois-je ici paraître ?
Dit l'animal chassé du paternel logis.
Holà ! madame la belette,
Que l'on déloge sans trompette,
Ou je vais avertir tous les rats du pays.
La dame au nez pointu répondit que la terre
Était au premier occupant.
C'était un beau sujet de guerre,
Qu'un logis où lui-même il n'entrait qu'en rampant !
Et quand ce serait un royaume,
Je voudrais bien savoir, dit-elle, quelle loi
En a pour toujours fait l'octroi
À Jean, fils ou neveu de Pierre ou de Guillaume,
Plutôt qu'à Paul, plutôt qu'à moi.
Jean lapin allégua la coutume et l'usage :
Ce sont, dit-il, leurs lois qui m'ont de ce logis
Rendu maître et seigneur, et qui, de père en fils,
L'ont de Pierre à Simon, puis à moi Jean, transmis.
Le premier occupant, est-ce une loi plus sage ?
Or bien, sans crier davantage,
Rapportons-nous, dit-elle, à Raminagrobis.
C'était un chat vivant comme un dévot ermite,
Un chat faisant la chattemite,
Un saint homme de chat, bien fourré, gros et gras,
Arbitre expert sur tous les cas.
Jean lapin pour juge l'agrée.
Les voilà tous deux arrivés
Devant sa majesté fourrée.
Grippeminaud leur dit : Mes enfants, approchez,
Approchez : je suis sourd, les ans en sont la cause.
L'un et l'autre approcha, ne craignant nulle chose.
Aussitôt qu'à portée il vit les contestants,
Grippeminaud le bon apôtre,
Jetant des deux côtés la griffe en même temps,
Mit les plaideurs d'accord en croquant l'un et l'autre.
Ceci ressemble fort aux débats qu'ont parfois
Les petits souverains se rapportant aux rois.


LES DEUX MULETS
Deux mulets cheminaient, l'un d'avoine chargé,
L'autre portant l'argent de la gabelle.
Celui-ci, glorieux d'une charge si belle,
N'eût voulu pour beaucoup en être soulagé.
Il marchait d'un pas relevé
Et faisait sonner sa sonnette ;
Quand l'ennemi se présentant,
Comme il en voulait à l'argent,
Sur le mulet du fisc une troupe se jette,
Le saisit au frein, et l'arrête.
Le mulet, en se défendant,
Se sent percer de coups ; il gémit, il soupire.
Est-ce donc là, dit-il, ce qu'on m'avait promis ?
Ce mulet qui me suit du danger se retire,
Et moi j'y tombe et je péris !
Ami, lui dit son camarade,
Il n'est pas toujours bon d'avoir un haut emploi :
Si tu n'avais servi qu'un meunier comme moi,
Tu ne serais pas si malade.
"""

print(f"{len(corpus)} caractères")
print(corpus[:236])

38331 caractères
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêt


In [23]:
# Le MiniLM du chapitre 1, repris tel quel (chaque ligne y est expliquee).
import torch.nn as nn


def get_device():
    """Renvoie le meilleur processeur disponible : cuda -> mps -> cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_device()
torch.manual_seed(42)

chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
block_size = 16


class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, 24)
        self.reseau = nn.Sequential(
            nn.Linear(block_size * 24, 192),
            nn.Tanh(),
            nn.Linear(192, vocab_size),
        )

    def forward(self, x):
        e = self.table(x)
        return self.reseau(e.flatten(1))


def fabriquer_batch(taille=64):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix]).to(device)
    y = data[ix + block_size].to(device)
    return x, y


model = MiniLM().to(device)
n_params = sum(p.numel() for p in model.parameters())
assert vocab_size == 81, f"le corpus du chapitre 1 compte 81 caracteres distincts, obtenu {vocab_size}"
assert n_params == 91_497, f"MiniLM compte 91 497 nombres, obtenu {n_params}"
print(f"MiniLM reconstruit : {n_params} nombres a apprendre, vocabulaire de {vocab_size} caracteres")

MiniLM reconstruit : 91497 nombres a apprendre, vocabulaire de 81 caracteres


In [24]:
def cross_entropy_100_maison(logits, cibles):
    """La chaine complete avec TON softmax : rien de PyTorch sauf exp, log, mean."""
    probas = torch.stack([softmax_maison(ligne) for ligne in logits])   # ton softmax, ligne par ligne
    p_bonnes = probas[range(len(cibles)), cibles]                       # la proba de la bonne reponse
    return -torch.log(p_bonnes).mean()                                  # -log, puis moyenne


# Etape 0 : poids au hasard. La loss doit valoir la loss du hasard pur, log(81).
x, y = fabriquer_batch()
logits = model(x)
loss_maison  = cross_entropy_100_maison(logits, y)
loss_pytorch = F.cross_entropy(logits, y)
print(f"etape 0 | loss maison = {loss_maison.item():.4f} | F.cross_entropy = {loss_pytorch.item():.4f}"
      f" | -log(1/81) = {math.log(81):.4f}")
assert torch.allclose(loss_maison, loss_pytorch, atol=1e-4), "les deux calculs doivent coincider"
assert 3.9 < loss_maison.item() < 5.0, "des poids aleatoires doivent donner une loss proche de log(81) = 4.39"

etape 0 | loss maison = 4.4595 | F.cross_entropy = 4.4595 | -log(1/81) = 4.3944


In [25]:
# Reentrainement express : la boucle du chapitre 1, 2 000 etapes.
optimiseur = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(2001):
    x, y = fabriquer_batch()
    loss = F.cross_entropy(model(x), y)
    optimiseur.zero_grad()
    loss.backward()
    optimiseur.step()
    if step % 500 == 0:
        print(f"etape {step:4d} | loss {loss.item():.2f}")

etape    0 | loss 4.42
etape  500 | loss 2.54
etape 1000 | loss 2.07
etape 1500 | loss 1.89
etape 2000 | loss 1.96


In [26]:
# Apres entrainement : les deux calculs doivent encore afficher les memes chiffres.
with torch.no_grad():
    x, y = fabriquer_batch()
    logits = model(x)
    loss_maison  = cross_entropy_100_maison(logits, y)
    loss_pytorch = F.cross_entropy(logits, y)

print(f"loss maison      : {loss_maison.item():.4f}")
print(f"F.cross_entropy  : {loss_pytorch.item():.4f}")
print(f"P moyenne donnee a la bonne reponse : e^-loss = {math.exp(-loss_pytorch.item()):.2f}")
assert torch.allclose(loss_maison, loss_pytorch, atol=1e-4), \
    f"ecart entre maison ({loss_maison.item():.6f}) et PyTorch ({loss_pytorch.item():.6f})"
assert loss_pytorch.item() < 2.5, f"apres 2 000 etapes la loss doit avoir fondu, obtenu {loss_pytorch.item():.2f}"
print("Bonus OK : ton softmax et ta cross-entropy retrouvent exactement la loss de PyTorch.")

loss maison      : 1.6033
F.cross_entropy  : 1.6033
P moyenne donnee a la bonne reponse : e^-loss = 0.20
Bonus OK : ton softmax et ta cross-entropy retrouvent exactement la loss de PyTorch.


In [27]:
# La recompense : generer, la fonction du chapitre 1, sur le modele reentraine.
itos = {i: c for c, i in stoi.items()}


def generer(prompt="\n", longueur=300, temperature=1.0):
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]
    sortie = []
    for _ in range(longueur):
        scores = model(torch.tensor([ctx]).to(device))       # les 81 logits (seule adaptation : .to(device))
        probas = F.softmax(scores / temperature, dim=-1)     # scores -> distribution, dosee (section 2)
        i = torch.multinomial(probas, num_samples=1).item()  # le tirage au sort (section 4)
        sortie.append(itos[i])
        ctx = ctx[1:] + [i]                                  # le gagnant rejoint le contexte
    return prompt + "".join(sortie)


with torch.no_grad():
    print(generer(prompt="La cigale ", longueur=300, temperature=0.8))

La cigale pencent trenge jouver jes ne la pres :
Ait pais qu'ol etor fais ant le me mesin on t mais en..
Un ces meus, carérine res lantête, yaî pant es vemait se los erise allaiente.
Quile t pause à joute
De defrisine et ceil, prant que sa det des en stor la ant pas en re shabpai.
Ois n'ut in pom ures de sagn


## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●).
Tu réécris de mémoire les quatre fonctions clés du chapitre, sans regarder la
leçon ci-dessus (c'est le jeu). Chaque cellule marquée `# TODO(toi)` contient un
trou ; complète-le, puis exécute la cellule de validation (`assert`) qui suit :
si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution
(`solutions/partie_1_etincelle/chapitre_05_un_peu_de_hasard_solution.ipynb`),
à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Le softmax à la main — niveau ●

Les deux gestes qui transforment des scores bruts (logits) en distribution de
probabilités. Réécris-les dans `ton_softmax`, sans recopier `softmax_maison` :
exponentielle d'abord, division par la somme ensuite.

In [34]:
def ton_softmax(scores):
    """Transforme des scores bruts (logits) en distribution de probabilites."""
    # TODO(toi) : les deux gestes du softmax.
    # 1) exponentie chaque score avec torch.exp : tout devient positif ;
    # 2) divise le resultat par la somme des exponentielles : le total vaut 1.
    exps  = torch.exp(scores)
    return exps / sum(exps)

In [37]:
# Validation du softmax : les valeurs calculees a la main dans le chapitre.
probas_ex = ton_softmax(scores)
assert isinstance(probas_ex, torch.Tensor), "ton_softmax doit renvoyer un tenseur"
attendu = torch.tensor([0.6590, 0.2424, 0.0986])   # e^2/11.21, e^1/11.21, e^0.1/11.21
assert torch.allclose(probas_ex, attendu, atol=1e-3), f"attendu {attendu.tolist()}, obtenu {probas_ex.tolist()}"
assert abs(probas_ex.sum().item() - 1.0) < 1e-6, "la somme des probabilites doit valoir 1"
assert (probas_ex > 0).all(), "toutes les probabilites doivent etre strictement positives"
assert torch.allclose(probas_ex, F.softmax(scores, dim=-1), atol=1e-6), "doit coincider avec F.softmax"
print("Softmax OK :", [round(p, 4) for p in probas_ex.tolist()], "| somme =", round(probas_ex.sum().item(), 6))

Softmax OK : [0.659, 0.2424, 0.0986] | somme = 1.0


### Exercice 2 · La molette temperature — niveau ●

Une seule ligne : divise les scores par `T` avant d'appeler `ton_softmax`.
$T < 1$ pique la distribution, $T > 1$ l'aplatit.

In [38]:
def ton_softmax_temperature(scores, T):
    """Le softmax avec sa molette : divise les scores par T avant le softmax."""
    # TODO(toi) : une seule ligne, en reutilisant ton_softmax.
    return ton_softmax(scores / T)

In [39]:
# Validation de la temperature : le tableau du chapitre, au dixieme de pourcent.
piquee  = ton_softmax_temperature(scores, T=0.5)
neutre  = ton_softmax_temperature(scores, T=1.0)
aplatie = ton_softmax_temperature(scores, T=2.0)
assert torch.allclose(piquee,  torch.tensor([0.8638, 0.1169, 0.0193]), atol=1e-3), f"T=0.5 : obtenu {piquee.tolist()}"
assert torch.allclose(neutre,  F.softmax(scores, dim=-1), atol=1e-5), "T=1 doit redonner le softmax tel quel"
assert torch.allclose(aplatie, torch.tensor([0.5017, 0.3043, 0.1940]), atol=1e-3), f"T=2 : obtenu {aplatie.tolist()}"
print("Temperature OK :")
for nom, T, p in [("piquee ", 0.5, piquee), ("neutre ", 1.0, neutre), ("aplatie", 2.0, aplatie)]:
    print(f"  T = {T} ({nom}) : " + "  ".join(f"{c} {v*100:4.1f} %" for c, v in zip("iou", p.tolist())))

Temperature OK :
  T = 0.5 (piquee ) : i 86.4 %  o 11.7 %  u  1.9 %
  T = 1.0 (neutre ) : i 65.9 %  o 24.2 %  u  9.9 %
  T = 2.0 (aplatie) : i 50.2 %  o 30.4 %  u 19.4 %


### Exercice 3 · La cross-entropy à la main — niveau ●●

La formule du chapitre, mot à mot, sur un batch de logits : softmax ligne par
ligne, lecture de la probabilité de la bonne réponse dans chaque ligne, puis
moins-log et moyenne.

In [40]:
def ta_cross_entropy(logits, cibles):
    """La formule du chapitre, mot a mot, sur un batch de logits."""
    # TODO(toi) : la formule en trois etapes.
    # 1) probas : F.softmax(logits, dim=1), chaque ligne devient une distribution ;
    # 2) p_bonnes : dans la ligne i, pioche la colonne cibles[i]
    #    (indexation avancee : probas[range(len(cibles)), cibles]) ;
    # 3) renvoie -torch.log(p_bonnes).mean() : moins-log, puis moyenne.
    probas = F.softmax(logits, dim=1)
    p_bonnes = probas[range(len(cibles)), cibles]
    return -torch.log(p_bonnes).mean()

In [41]:
# Validation de la cross-entropy : le batch de trois exemples du chapitre.
logits_ex = torch.tensor([[2.0, 1.0, 0.1],     # exemple 1 : la bonne reponse est i (indice 0)
                          [0.2, 2.3, 0.5],     # exemple 2 : la bonne reponse est o (indice 1)
                          [1.0, 1.0, 1.0]])    # exemple 3 : la bonne reponse est u (indice 2)
cibles_ex = torch.tensor([0, 1, 2])            # les indices des bonnes reponses

loss_ex = ta_cross_entropy(logits_ex, cibles_ex)
assert isinstance(loss_ex, torch.Tensor), "ta_cross_entropy doit renvoyer un tenseur"
assert abs(loss_ex.item() - 0.5895) < 1e-3, f"loss attendue 0.5895, obtenue {loss_ex.item():.4f}"
assert torch.allclose(loss_ex, F.cross_entropy(logits_ex, cibles_ex), atol=1e-5), \
    "doit coincider avec F.cross_entropy (qui contient deja le softmax)"
print(f"Cross-entropy OK : maison = {loss_ex.item():.4f} | F.cross_entropy = {F.cross_entropy(logits_ex, cibles_ex).item():.4f}")

Cross-entropy OK : maison = 0.5895 | F.cross_entropy = 0.5895


### Exercice 4 · Le tirage au sort — niveau ●●●

La roue de loterie dépliée : découpe le segment $[0, 1)$ en zones dont les
tailles sont les probabilités (les bornes sont les **sommes cumulées**), lance
une fléchette uniforme, et renvoie la zone où elle tombe.

In [42]:
def tirer_au_sort(probas):
    """Tire un indice au sort, proportionnellement aux probabilites."""
    r = random.random()             # la flechette : uniforme entre 0 et 1
    cumul = 0.0
    # TODO(toi) : parcours les candidats avec enumerate(probas) ;
    # 1) ajoute chaque probabilite a cumul (la borne droite de la zone du candidat) ;
    # 2) des que r < cumul, la flechette est tombee dans la zone : renvoie l'indice.
    for i, p in enumerate(probas):
        cumul += p
        if r < cumul:
            return i
    return len(probas) - 1          # filet de securite (arrondis flottants)

In [43]:
# Validation du tirage : cas certains, puis 10 000 tirages comptes.
assert tirer_au_sort([1.0, 0.0, 0.0]) == 0, "toute la masse sur i : i doit toujours sortir"
assert tirer_au_sort([0.0, 0.0, 1.0]) == 2, "toute la masse sur u : u doit toujours sortir"

random.seed(0)                                     # flechettes reproductibles
tirages_ex = [tirer_au_sort([0.659, 0.242, 0.099]) for _ in range(10_000)]
comptes_ex = [tirages_ex.count(i) for i in range(3)]
assert comptes_ex == [6597, 2460, 943], f"avec random.seed(0), comptes attendus [6597, 2460, 943], obtenus {comptes_ex}"
for c, n in zip("iou", comptes_ex):
    print(f"« {c} » : {n:5d} tirages sur 10 000  ({n / 100:.1f} %)")
print("Echantillonnage OK : le hasard obeit aux probabilites (66 % / 24 % / 10 % environ).")

« i » :  6597 tirages sur 10 000  (66.0 %)
« o » :  2460 tirages sur 10 000  (24.6 %)
« u » :   943 tirages sur 10 000  (9.4 %)
Echantillonnage OK : le hasard obeit aux probabilites (66 % / 24 % / 10 % environ).


## Verdict

Quatre validations vertes : les trois mystères sont tombés. Tu sais transformer
des scores en probabilités, mesurer une prédiction qui parle en probabilités, et
tirer le caractère suivant au sort. Et le bonus l'a prouvé sur pièces : les
fonctions maison calculent, au dix-millième près, la même loss que celle qui a
entraîné MiniLM au chapitre 1.

Dernier geste pour clore la Partie I : rouvre le notebook du chapitre 1 et relis
le programme en entier, ligne à ligne. Plus une seule zone d'ombre.

Retour au livre pour la suite : *Chapitre 6 · Le neurone et le réseau*.